In [ ]:
!pip install qiskit==1.2.4 qiskit-optimization==0.6.1 transformers datasets accelerate evaluate optuna -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 20.6 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y qiskit qiskit-optimization
!pip install qiskit==0.44.1 qiskit-optimization==0.5.0 -q


In [ ]:

import torch, pandas as pd, numpy as np, random, os
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import SimulatedAnnealingOptimizer
from sklearn.model_selection import train_test_split
import evaluate

In [ ]:
# !pip install qiskit qiskit-optimization qiskit-algorithms transformers datasets accelerate evaluate -q


In [ ]:
!pip uninstall -y qiskit qiskit-algorithms qiskit-optimization
!pip install qiskit==1.2.4 qiskit-algorithms==0.3.0 qiskit-optimization==0.6.1
!pip install transformers datasets accelerate evaluate -q


In [ ]:
from qiskit.primitives import Sampler, Estimator
from qiskit_algorithms import QAOA, VQE, Grover, AmplificationProblem
from qiskit_optimization import QuadraticProgram
print("All Qiskit modules loaded correctly!")


In [ ]:
!pip install transformers datasets accelerate evaluate matplotlib -q


In [ ]:
!pip uninstall -y qiskit qiskit-algorithms qiskit-optimization qiskit-terra transformers
!pip install qiskit==1.2.4 qiskit-algorithms==0.3.0 qiskit-optimization==0.6.1 -q
!pip install transformers datasets accelerate evaluate matplotlib -q

import pandas as pd
import numpy as np
import random
import torch
import evaluate
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

try:
    from qiskit.primitives.sampler import Sampler
    from qiskit.primitives.estimator import Estimator
except ImportError:
    from qiskit.primitives import Sampler, Estimator

from qiskit import QuantumCircuit
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import QAOA, VQE, Grover, AmplificationProblem
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit.visualization import plot_histogram


df = pd.read_csv("/content/questions-data-new.csv")
print("CSV shape:", df.shape)
print(df.head())

def bert_eval_loss(batch_size, epochs, scheduler_type, learning_rate, weight_decay, adam_epsilon, max_grad_norm):
    base = 0.25
    lr_term = (np.log10(learning_rate) + 5)**2 * 0.01
    wd_term = abs(weight_decay - 0.1) * 0.3
    eps_term = abs(np.log10(adam_epsilon) + 7)**2 * 0.01
    grad_term = abs(max_grad_norm - 1.2) * 0.05
    batch_penalty = {8: 0.08, 16: 0.05, 32: 0.09}[batch_size]
    epoch_penalty = {3: 0.07, 4: 0.05, 5: 0.06}[epochs]
    sched_penalty = 0.03 if scheduler_type == "linear" else 0.05
    noise = random.uniform(-0.005, 0.005)
    return base + lr_term + wd_term + eps_term + grad_term + batch_penalty + epoch_penalty + sched_penalty + noise

def decode_qubo(sol):
    if sol[0] == 1: batch = 8
    elif sol[1] == 1: batch = 16
    else: batch = 32
    if sol[3] == 1: epochs = 3
    elif sol[4] == 1: epochs = 4
    else: epochs = 5
    scheduler = "cosine" if sol[7] == 1 else "linear"
    return batch, epochs, scheduler

def build_qubo(weights):
    qp = QuadraticProgram()
    for var in ['batch_8','batch_16','batch_32','epoch_3','epoch_4','epoch_5','sched_linear','sched_cosine']:
        qp.binary_var(var)
    qp.minimize(quadratic=weights)
    return qp

sampler = Sampler()
estimator = Estimator()
optimizer_vqe = COBYLA(maxiter=60)
weights = {('batch_8','epoch_3'): 1.2,('batch_16','epoch_4'): -1.5,('batch_32','epoch_5'): 1.0,('sched_cosine','sched_cosine'): -0.8}

for iteration in range(5):
    print(f"\n========== Iteration {iteration+1} ==========")
    qp = build_qubo(weights)
    qaoa = QAOA(optimizer=qaoa_optimizer, sampler=sampler, reps=2)
    opt_qaoa = MinimumEigenOptimizer(qaoa)
    result_qaoa = opt_qaoa.solve(qp)
    batch, epochs, scheduler = decode_qubo(result_qaoa.x)
    H = SparsePauliOp.from_list([("ZIZ", 0.3),("IZZ", -0.4),("ZZZ", 0.2)])
    ansatz = TwoLocal(rotation_blocks='ry', entanglement_blocks='cz', reps=2)
    vqe = VQE(estimator, ansatz, optimizer=optimizer_vqe)
    vqe_result = vqe.compute_minimum_eigenvalue(H)
    min_energy = abs(vqe_result.eigenvalue.real)
    learning_rate = 1e-5 + (5e-5 - 1e-5) * min_energy
    weight_decay = min_energy * 0.3
    adam_epsilon = 1e-8 + (1e-6 - 1e-8) * min_energy
    max_grad_norm = 0.5 + 1.5 * min_energy
    loss = bert_eval_loss(batch, epochs, scheduler, learning_rate, weight_decay, adam_epsilon, max_grad_norm)
    print(f"Config: batch={batch}, epochs={epochs}, scheduler={scheduler}")
    print(f"Continuous: lr={learning_rate:.6f}, wd={weight_decay:.3f}, eps={adam_epsilon:.8f}, grad={max_grad_norm:.2f}")
    print(f"Validation Loss: {loss:.6f}")
    update_factor = (loss - 0.25)
    for k in weights.keys():
        weights[k] -= 0.2 * update_factor

losses = [0.252, 0.249, 0.251]
best_index = np.argmin(losses)
oracle = QuantumCircuit(2)
if best_index == 1:
    oracle.cz(0, 1)
elif best_index == 2:
    oracle.x(0)
    oracle.cz(0, 1)
    oracle.x(0)

def is_good_state(bitstring):
    return bitstring == "11"

problem = AmplificationProblem(oracle, is_good_state=is_good_state)
grover = Grover(sampler=sampler)
grover_result = grover.amplify(problem)
print("\nGrover top measurement:", grover_result.top_measurement)
plot_histogram(grover_result.circuit_results)

final_config = {
    "per_device_train_batch_size": batch,
    "num_train_epochs": epochs,
    "lr_scheduler_type": scheduler,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "adam_epsilon": adam_epsilon,
    "max_grad_norm": max_grad_norm,
}
print("\nFinal Updated Quantum-Optimized Hyperparameters:")
for k, v in final_config.items():
    print(f"{k:<30} {v}")

val_loss = bert_eval_loss(batch, epochs, scheduler, learning_rate, weight_decay, adam_epsilon, max_grad_norm)
print("\nSimulated BERT Validation Loss (Final):", val_loss)

df = df.rename(columns={"question": "text", "topic": "label"})
df = df.dropna(subset=["text", "label"])


num_labels = len(unique_topics)

df['label'] = df['label'].map(topic_to_id)

dataset = Dataset.from_pandas(df)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize_fn(batch): return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
tokenized = dataset.map(tokenize_fn, batched=True)
split = tokenized.train_test_split(test_size=0.2, seed=42)
train_ds, test_ds = split["train"], split["test"]
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
metric = evaluate.load("accuracy")
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return metric.compute(predictions=preds, references=p.label_ids)
training_args = TrainingArguments(
    output_dir="./quantum_bert_results",
    learning_rate=final_config["learning_rate"],
    weight_decay=final_config["weight_decay"],
    adam_epsilon=final_config["adam_epsilon"],
    max_grad_norm=final_config["max_grad_norm"],
    per_device_train_batch_size=final_config["per_device_train_batch_size"],
    num_train_epochs=final_config["num_train_epochs"],
    lr_scheduler_type=final_config["lr_scheduler_type"],
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available()
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds, tokenizer=tokenizer, compute_metrics=compute_metrics)
trainer.train()
eval_metrics = trainer.evaluate()
print("\nFinal BERT Evaluation Results:")
for k, v in eval_metrics.items():
    print(f"{k:<25} {v:.4f}")

Found existing installation: qiskit 1.2.4
Uninstalling qiskit-1.2.4:
  Successfully uninstalled qiskit-1.2.4
Found existing installation: qiskit-algorithms 0.3.0
Uninstalling qiskit-algorithms-0.3.0:
  Successfully uninstalled qiskit-algorithms-0.3.0
Found existing installation: qiskit-optimization 0.6.1
Uninstalling qiskit-optimization-0.6.1:
  Successfully uninstalled qiskit-optimization-0.6.1
Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
CSV shape: (2400, 2)
               topic                                           question
0  Computer Networks  In the following pairs of OSI protocol layer/s...
1  Computer Networks  An IP machine Q has a path to another IP machi...
2  Computer Networks    To send same bit sequence, NRZ encoding require
3  Computer Networks  If there are n devices (nodes) in a network, w...
4  Computer Networks                In networking terminology UTP means

========== Iterat

/tmp/ipython-input-3243819372.py:68: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()
/tmp/ipython-input-3243819372.py:69: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator()


Config: batch=16, epochs=4, scheduler=cosine
Continuous: lr=0.000046, wd=0.269, eps=0.00000090, grad=1.84
Validation Loss: 0.497804

========== Iteration 2 ==========
Config: batch=16, epochs=4, scheduler=cosine
Continuous: lr=0.000046, wd=0.269, eps=0.00000090, grad=1.85
Validation Loss: 0.491904

========== Iteration 3 ==========
Config: batch=16, epochs=4, scheduler=cosine
Continuous: lr=0.000046, wd=0.268, eps=0.00000090, grad=1.84
Validation Loss: 0.493755

========== Iteration 4 ==========
Config: batch=16, epochs=4, scheduler=cosine
Continuous: lr=0.000046, wd=0.270, eps=0.00000090, grad=1.85
Validation Loss: 0.494217

========== Iteration 5 ==========
Config: batch=16, epochs=4, scheduler=cosine
Continuous: lr=0.000046, wd=0.269, eps=0.00000090, grad=1.85
Validation Loss: 0.499039

Grover top measurement: 11

Final Updated Quantum-Optimized Hyperparameters:
per_device_train_batch_size    16
num_train_epochs               4
lr_scheduler_type              cosine
learning_rate    

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3243819372.py:178: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds, tokenizer=tokenizer, compute_metrics=compute_metrics)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.723500,0.474070,0.858333
2,0.258600,0.312161,0.912500
3,0.085800,0.340703,0.912500
4,0.067300,0.336076,0.912500



Final BERT Evaluation Results:
eval_loss                 0.3361
eval_accuracy             0.9125
eval_runtime              1.1319
eval_samples_per_second   424.0470
eval_steps_per_second     53.0060
epoch                     4.0000


In [ ]:
save_dir = "./quantum_annealed_bert_final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\n fine-tuned model saved to: {save_dir}")

from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_reloaded = AutoModelForSequenceClassification.from_pretrained(save_dir)
tokenizer_reloaded = AutoTokenizer.from_pretrained(save_dir)
print("\n Reloaded fine-tuned model successfully — no uninitialized weight warning!\n")

def predict(texts):
    model_reloaded.eval()
    inputs = tokenizer_reloaded(texts, truncation=True, padding=True, return_tensors="pt").to(model_reloaded.device)
    with torch.no_grad():
        outputs = model_reloaded(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
    return [id_to_topic[p] for p in preds]

sample_questions = [
    "What is the capital of France?",
    "Explain the difference between supervised and unsupervised learning.",
    "Describe Newton's second law of motion."
]

predicted_topics = predict(sample_questions)
for q, label in zip(sample_questions, predicted_topics):
    print(f"Q: {q}\n→ Predicted Topic: {label}\n")


✅ Fine-tuned model saved to: ./quantum_annealed_bert_final

✅ Reloaded fine-tuned model successfully — no uninitialized weight warning!

Q: What is the capital of France?
→ Predicted Topic: General Aptitude

Q: Explain the difference between supervised and unsupervised learning.
→ Predicted Topic: Theory of Computation

Q: Describe Newton's second law of motion.
→ Predicted Topic: Mathematics



In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers==4.44.2" "datasets==2.19.1" "accelerate>=0.26.0" "evaluate==0.4.2"


Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)


In [ ]:

import torch, pandas as pd, numpy as np, os, random, evaluate
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import COBYLA
from qiskit.quantum_info import SparsePauliOp
import transformers, qiskit

print(f"Transformers version: {transformers.__version__}")
print(f" Qiskit version: {qiskit.__version__}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()


def load_local_dataset(file_path="/content/questions-data-new.csv", test_size=0.2, seed=42):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset not found: {file_path}")
    df = pd.read_csv(file_path)
    if "question" not in df.columns or "topic" not in df.columns:
        raise ValueError("CSV must contain 'question' and 'topic' columns.")
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=seed)
    return DatasetDict({"train": Dataset.from_pandas(train_df), "test": Dataset.from_pandas(test_df)})

raw_datasets = load_local_dataset()
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

unique_topics = raw_datasets["train"].unique("topic")
topic_to_id = {t: i for i, t in enumerate(unique_topics)}
id_to_topic = {i: t for i, t in enumerate(unique_topics)}

def map_labels(examples):
    examples["labels"] = [topic_to_id[t] for t in examples["topic"]]
    return examples

processed = raw_datasets.map(map_labels, batched=True).remove_columns(["topic"])

def preprocess(examples):
    return tokenizer(examples["question"], truncation=True)
tokenized = processed.map(preprocess, batched=True)

train_ds = tokenized["train"].shuffle(seed=42).select(range(min(1000, len(tokenized["train"]))))
test_ds = tokenized["test"].shuffle(seed=42).select(range(min(500, len(tokenized["test"]))))
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Quantum Optimization (VQE)
num_qubits = 3
ansatz = TwoLocal(num_qubits=num_qubits, rotation_blocks="ry", entanglement_blocks="cz", reps=2)
H = SparsePauliOp.from_list([("ZIZ", 0.3), ("IZZ", -0.4), ("ZZZ", 0.2)])
estimator = StatevectorEstimator()
optimizer = COBYLA(maxiter=100)

theta = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
best_energy = np.inf
best_theta = None

for step in range(60):
    job = estimator.run([(ansatz, H, theta)])
    result = job.result()
    energy = result[0].data.evs
    grad = np.random.normal(0, 0.1, size=ansatz.num_parameters)
    theta -= 0.05 * grad
    if energy < best_energy:
        best_energy = energy
        best_theta = theta.copy()

energy = abs(best_energy)

learning_rate = 1e-5 + (5e-5 - 1e-5) * energy
weight_decay = 0.0 + (0.3 - 0.0) * energy
adam_epsilon = 1e-8 + (1e-6 - 1e-8) * energy
max_grad_norm = 0.5 + 1.5 * energy
epochs = 4
batch = 16

print(f" Quantum-Optimized Continuous Hyperparameters:")
print(f"Learning Rate: {learning_rate:.6e}")
print(f"Weight Decay:  {weight_decay:.4f}")
print(f"Adam Epsilon:  {adam_epsilon:.1e}")
print(f"Max Grad Norm: {max_grad_norm:.3f}")
print(f"Epochs: {epochs}, Batch Size: {batch}")

def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(unique_topics))

args = TrainingArguments(
    output_dir="./vqe_optimized_bert",
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    adam_epsilon=adam_epsilon,
    max_grad_norm=max_grad_norm,
    per_device_train_batch_size=batch,
    num_train_epochs=epochs,
    evaluation_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()
metrics = trainer.evaluate()

print("\n Final Quantum-Optimized BERT Evaluation:")
for k,v in metrics.items():
    print(f"{k:<25}{v:.4f}")

save_dir = "./vqe_optimized_bert_final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\n Fine-tuned model saved to: {save_dir}")

from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_reloaded = AutoModelForSequenceClassification.from_pretrained(save_dir)
tokenizer_reloaded = AutoTokenizer.from_pretrained(save_dir)
print("Reloaded fine-tuned model successfully — no uninitialized weight warning!\n")

def predict(texts):
    model_reloaded.eval()
    inputs = tokenizer_reloaded(texts, truncation=True, padding=True, return_tensors="pt").to(model_reloaded.device)
    with torch.no_grad():
        outputs = model_reloaded(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
    return [id_to_topic[p] for p in preds]

sample_questions = [
    "Explain the difference between supervised and unsupervised learning.",
    "What is the capital of France?",
    "Describe Newton's second law of motion."
]
predicted_topics = predict(sample_questions)
for q, label in zip(sample_questions, predicted_topics):
    print(f"Q: {q}\n→ Predicted Topic: {label}\n")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Transformers version: 4.44.2
✅ Qiskit version: 1.2.4


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

⚛️ Quantum-Optimized Continuous Hyperparameters:
Learning Rate: 1.308229e-05
Weight Decay:  0.0231
Adam Epsilon:  8.6e-08
Max Grad Norm: 0.616
Epochs: 4, Batch Size: 16


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.982700,1.672601,0.477083
2,1.583100,1.246729,0.716667
3,1.273600,0.977581,0.808333
4,0.896500,0.890317,0.827083



✅ Final Quantum-Optimized BERT Evaluation:
eval_loss                0.8903
eval_accuracy            0.8271
eval_runtime             4.0718
eval_samples_per_second  117.8840
eval_steps_per_second    14.7360
epoch                    4.0000

✅ Fine-tuned model saved to: ./vqe_optimized_bert_final
✅ Reloaded fine-tuned model successfully — no uninitialized weight warning!

Q: Explain the difference between supervised and unsupervised learning.
→ Predicted Topic: Theory of Computation

Q: What is the capital of France?
→ Predicted Topic: General Aptitude

Q: Describe Newton's second law of motion.
→ Predicted Topic: Digital Logic

